<a href="https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


In [13]:
content_schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
print(content_schema.to_string(index=False))

               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_count      BIGINT  YES None    None  None
            url_char_count      BIGINT  YES None    None  None
      content_created_date        DATE  YES None    None  None
      content_updated_date        DATE  YES None    None  None
              content_type     VARCHAR  YES None    None  None
             search_volume      BIGINT  YES None    None  None
               competition      DOUBLE  YES None    None  None
         competition_level     VARCHAR  YES None    None  None
                       cpc      DOUBLE  YES None    None  None
               main_intent     VARCHAR  YES None    Non

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [14]:
TABLES['fact_daily_sample'] = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"
print("Added fact_daily_sample to TABLES")

Added fact_daily_sample to TABLES


In [15]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily_sample']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev60,
               STDDEV(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS position_volatility
        FROM {TABLES['fact_daily_sample']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev60 >= 150
    )
    SELECT *, imp_prev60 / 2.0 AS imp_prev60_avg30
    FROM windowed
""").df()

features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev60_avg30']).astype(int)
print(f"{len(features):,} content items")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0 content items


In [16]:
sample_check = con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest, COUNT(*) AS n
    FROM {TABLES['fact_daily_sample']}
""").df()
print(sample_check)

    earliest     latest         n
0 2026-06-01 2026-06-30  11694072


In [17]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily_sample']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last15,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev15,
               STDDEV(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_avg_position END) AS position_volatility
        FROM {TABLES['fact_daily_sample']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 30 DAY
        GROUP BY 1, 2
        HAVING imp_prev15 >= 50
    )
    SELECT *
    FROM windowed
""").df()

features['is_declining'] = (features['imp_last15'] < 0.8 * features['imp_prev15']).astype(int)
print(f"{len(features):,} content items")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

99,893 content items


,client_hash_id,content_hash_id,imp_last15,imp_prev15,position_volatility,is_declining
0,client_62f4a7e64f5e0096,content_e4376489e7635039,1883.0,2193.0,0.964615,0
1,client_62f4a7e64f5e0096,content_ec8d97984dbf4376,256.0,1726.0,8.677069,1
2,client_62f4a7e64f5e0096,content_531a5af64aefa6f7,3438.0,2113.0,1.275710,0
3,client_62f4a7e64f5e0096,content_ba0d2a28965016f9,7410.0,7784.0,0.561755,0
4,client_62f4a7e64f5e0096,content_4621053ad3f9069d,38.0,61.0,26.814637,1


In [18]:
import pandas as pd
content_meta = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_type, main_intent,
           competition_level, word_count, content_created_date,
           last_optimized_date, optimization_eligible_date
    FROM {TABLES['dim_content']}
""").df()

data = features.merge(content_meta, on=['client_hash_id', 'content_hash_id'], how='left')

print(f"Missing content_type: {data['content_type'].isna().sum()} / {len(data)}")
print(f"Missing word_count: {data['word_count'].isna().sum()} / {len(data)}")

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')
data['competition_level'] = data['competition_level'].fillna('unknown')
data['word_count'] = data['word_count'].fillna(data['word_count'].median())

content_type_dummies = pd.get_dummies(data['content_type'], prefix='ctype')
data = pd.concat([data, content_type_dummies], axis=1)

print(f"\nFinal feature vector shape: {data.shape}")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Missing content_type: 0 / 99893
Missing word_count: 18242 / 99893

Final feature vector shape: (99893, 16)


,client_hash_id,content_hash_id,imp_last15,imp_prev15,position_volatility,is_declining,content_type,main_intent,competition_level,word_count,content_created_date,last_optimized_date,optimization_eligible_date,ctype_comparison article,ctype_feedly article,ctype_keyword article
0,client_62f4a7e64f5e0096,content_e4376489e7635039,1883.0,2193.0,0.964615,0,keyword article,informational,LOW,2825,2025-12-18,2026-06-11,2026-07-26,False,False,True
1,client_62f4a7e64f5e0096,content_ec8d97984dbf4376,256.0,1726.0,8.677069,1,keyword article,informational,LOW,2321,2025-12-18,2026-06-09,2026-07-24,False,False,True
2,client_62f4a7e64f5e0096,content_531a5af64aefa6f7,3438.0,2113.0,1.275710,0,keyword article,informational,LOW,2409,2025-12-18,2026-06-11,2026-07-26,False,False,True
3,client_62f4a7e64f5e0096,content_ba0d2a28965016f9,7410.0,7784.0,0.561755,0,keyword article,informational,LOW,2264,2025-12-18,2026-06-22,2026-08-06,False,False,True
4,client_62f4a7e64f5e0096,content_4621053ad3f9069d,38.0,61.0,26.814637,1,keyword article,informational,LOW,2793,2025-12-18,NaT,NaT,False,False,True


Built a feature vector by joining engineered daily signals (imp_last15, imp_prev15, position_volatility, from fact_daily_sample due to a slow query against the full fact_daily table, scaled the window to 15-vs-15 days to fit the sample's 30-day coverage) with content-level metadata from dim_content (content_type, main_intent, competition_level, word_count). Missing categorical values were filled with 'unknown' rather than dropped, since 0 rows were missing content_type but 18,242 of 99,893 (~18%) were missing word_count, a meaningful gap worth tracking, not silently imputing away. content_type was one-hot encoded into dummy columns for model use.

In [19]:
opt_coverage = con.sql(f"""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN last_optimized_date IS NOT NULL THEN 1 ELSE 0 END) AS has_last_optimized,
        SUM(CASE WHEN optimization_eligible_date IS NOT NULL THEN 1 ELSE 0 END) AS has_eligible_date
    FROM {TABLES['dim_content']}
""").df()
print(opt_coverage)

    total  has_last_optimized  has_eligible_date
0  519606             45396.0            45396.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature	Meaning	Missing handling	Available before prediction?
imp_last15 / imp_prev15	Impressions in recent vs. prior 15-day half	Required by HAVING filter, no missing	Yes, purely historical
position_volatility	Std dev of daily ranking position, recent 15 days	Possible if page had <2 days of data (rare), not filled, would need checking	Yes
content_type, main_intent, competition_level	Categorical content metadata	Filled with 'unknown'	Yes, set at content creation
word_count	Content length	18% missing, filled with median	Yes, static property
last_optimized_date, optimization_eligible_date	When content was last optimized / became eligible	Only 8.7% of content has these filled	Suspect — may reflect a decision already made, needs leakage check in Section 3

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [20]:
data['has_optimization_record'] = data['last_optimized_date'].notna().astype(int)

leakage_test = data.groupby('has_optimization_record')['is_declining'].agg(['mean', 'count'])
print("Declining rate by whether an optimization record exists:")
print(leakage_test)

# Also check: does the optimization date fall AFTER our observation window? (pure future leakage)
future_check = data[data['last_optimized_date'].notna()].copy()
future_check['optimized_after_window'] = (
    pd.to_datetime(future_check['last_optimized_date']) > pd.to_datetime('2026-06-30')
)
print(f"\nOf {len(future_check)} rows with an optimization date, "
      f"{future_check['optimized_after_window'].sum()} were optimized AFTER our data window ends")

Declining rate by whether an optimization record exists:
                             mean  count
has_optimization_record                 
0                        0.483449  62292
1                        0.540198  37601

Of 37601 rows with an optimization date, 4621 were optimized AFTER our data window ends


Tested the suspected leakage from Section 2 directly. Pages with an optimization record on file decline at 54.0% vs. 48.3% for pages without one (37,601 vs. 62,292 rows), confirming a real correlation between optimization activity and the decline label. More critically, of the 37,601 rows with an optimization date, 4,621 (12.3%) have a date that falls after our prediction window ends (2026-06-30), meaning the feature would literally encode outcome information from the future for those rows. This is direct, confirmed leakage: last_optimized_date and optimization_eligible_date are excluded from the feature set entirely, not just flagged as suspect.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

What I excluded and why, pulling together everything we've found across all four sections:

last_optimized_date, optimization_eligible_date: confirmed leakage (Section 3), 12.3% of dated rows fall after the prediction window, directly encoding future outcome information.
ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other: extreme sparsity (well under 0.1% of rows have any AI referral traffic, verified in ML-04), not usable as a general feature.
client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available: data-quality flags about coverage, not signals about the page itself.
ga4_* and channel-split sessions_* columns: only 3.6% of daily rows have GA4 data available (verified in ML-04), using them would silently restrict the model to a tiny, non-representative subset of clients.
client_hash_id, content_hash_id: excluded as model inputs (kept only as join/context keys), including raw IDs would let the model memorize specific pages/clients instead of learning a generalizable pattern.
gsc_sum_position: reconsidered in ML-04 after verification showed only 0.07 correlation with gsc_avg_position (not actually redundant), kept as a feature after all, noted here as a correction from an earlier assumption.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.